# Analisis del clasificador - Ejercicio 2

Este notebook evalua el comportamiento del clasificador usando las columnas categoria_esperada y categoria_predicha, incorporando contexto multilabel para interpretar errores.

In [ ]:
import ast
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('Ejercicio 2/sonia_routing_dataset.csv')
df.head()

In [ ]:
accuracy = accuracy_score(df['categoria_esperada'], df['categoria_predicha'])
print(f'Accuracy global: {accuracy:.4f}')

report = classification_report(df['categoria_esperada'], df['categoria_predicha'], output_dict=True)
pd.DataFrame(report).T

In [ ]:
labels = sorted(df['categoria_esperada'].unique())
cm = confusion_matrix(df['categoria_esperada'], df['categoria_predicha'], labels=labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Matriz de confusion')
plt.xlabel('Predicha')
plt.ylabel('Esperada')
plt.tight_layout()
plt.show()

In [ ]:
errors = df[df['categoria_esperada'] != df['categoria_predicha']].copy()
errors[['id', 'texto_usuario', 'categoria_esperada', 'categoria_predicha', 'multilabel_referencia', 'explicacion_referencia']].head(20)

In [ ]:
def parse_multilabel(value):
    try:
        parsed = ast.literal_eval(value)
        return [str(v).strip() for v in parsed]
    except Exception:
        return []

errors['multilabel_list'] = errors['multilabel_referencia'].apply(parse_multilabel)
errors['pred_in_multilabel'] = errors.apply(lambda r: r['categoria_predicha'] in r['multilabel_list'], axis=1)

summary = errors['pred_in_multilabel'].value_counts(dropna=False).rename_axis('prediccion_valida_en_multilabel').reset_index(name='casos')
summary

## Conclusiones

1. La metrica principal se calcula sobre categoria_esperada vs categoria_predicha.
2. El analisis multilabel permite distinguir errores de ruteo estricto vs predicciones plausibles en flujos compuestos.
3. Priorizar mejoras en las clases con peor precision/recall y en los pares de confusion mas frecuentes.